<a href="https://colab.research.google.com/github/jothiprakasam/llm/blob/main/llm/course/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
pip install -U datasets huggingface_hub transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [1]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 5.1.0
huggingface_hub: 1.33.0


In [4]:
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


# Load a small subset of the dataset
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

# Use only 100 training examples and 50 validation examples
small_train = raw_datasets["train"].select(range(100))
small_validation = raw_datasets["validation"].select(range(50))

# Tokenize the subsets
def tokenize_function(example):
    return tokenizer(
        example["sentence1"],
        example["sentence2"],
        truncation=True
    )

tokenized_train = small_train.map(
    tokenize_function, batched=True
)
tokenized_validation = small_validation.map(
    tokenize_function, batched=True
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Quick training configuration
training_args = TrainingArguments(
    output_dir="quick-bert-demo",
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    max_steps=10,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    report_to="none",
)

# Initialize the model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Train for only 10 steps
trainer.train()

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,0.784219
2,0.711058
3,0.793542
4,0.662868
5,0.634948
6,0.717523
7,0.664220
8,0.632048
9,0.686443
10,0.689062


TrainOutput(global_step=10, training_loss=0.6975930035114288, metrics={'train_runtime': 46.3416, 'train_samples_per_second': 1.726, 'train_steps_per_second': 0.216, 'total_flos': 2857221616800.0, 'train_loss': 0.6975930035114288, 'epoch': 0.7692307692307693})

In [5]:
import evaluate
import numpy as np

metric = evaluate.load("glue", "mrpc")

predictions = trainer.predict(tokenized_validation)

predicted_labels = np.argmax(
    predictions.predictions, axis=-1
)

results = metric.compute(
    predictions=predicted_labels,
    references=predictions.label_ids
)

print(results)

{'accuracy': 0.6, 'f1': 0.75}
